# Settings, stability and cost

The appendices on fitting settings, stability and convergence, and encoding, decoding and memory (Tables 20-25).

The settings of the two recipes are fields of their YAML files; the stability checks use the stored fits and a sweep of the final Gauss-Newton refinement; the costs were measured with one job per dedicated GPU.

In [1]:
import pathlib
import sys

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO = pathlib.Path.cwd()
while not (REPO / "neugk").exists():  # the repository root
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
RESULTS = REPO / "neugk/gyrosplat/results"
CONFIGS = REPO / "configs/gyrosplat"

from neugk.gyrosplat.eval.paper import experiments as X
from neugk.gyrosplat.eval.paper import load_experiments, load_rows

rows = load_rows(RESULTS / "rows/gkw_rows.jsonl")
exp = load_experiments(RESULTS / "experiments")
FIGURES = RESULTS / "figures"
FIGURES.mkdir(exist_ok=True)

from neugk.gyrosplat.config import load_config

## The recipe settings

The fields of `configs/gyrosplat/recipe/gkw_302x.yaml` and `gkw_142x.yaml` behind the settings of the appendix (the two recipes differ only in their size).

In [2]:
fields = [("n_equiv", "atom equivalents"), ("fit.steps", "field fit steps"), ("fit.polish_steps", "polish steps"),
          ("physics.epochs", "potential fit steps"), ("carriers.tied_k", "Gabor splats per group"),
          ("carriers.tail_bins", "carrier seed modes"), ("carriers.sigma_y_cells", "group width along y (cells)"),
          ("carriers.routed_steps", "joint fit steps"), ("carriers.routed_steps_hard", "joint fit steps, placement-bound"),
          ("carriers.hard_ratio", "gradient-ratio threshold"), ("carriers.flux_gate", "flux gate"),
          ("varpro.ridge", "amplitude ridge (x mean Gram diagonal)"), ("gn.steps", "Gauss-Newton steps"),
          ("gn.mu", "Gauss-Newton damping"), ("gn.carriers_only", "Gauss-Newton on the Gabor amplitudes only"),
          ("baselines.objective_warmup", "refined codes: Adam steps"),
          ("baselines.objective_relative_damping", "refined codes: damping (x max diag JJ^T)"), ("band", "flux modes")]
cfgs = {t: load_config(CONFIGS / f"recipe/gkw_{t}x.yaml") for t in (302, 142)}

def get(cfg, path):
    for part in path.split("."):
        cfg = getattr(cfg, part)
    return cfg

lines = ["| setting | field | 302× | 142× |", "|---|---|---|---|"]
for path, title in fields:
    lines.append(f"| {title} | `{path}` | {get(cfgs[302], path)} | {get(cfgs[142], path)} |")
display(Markdown("\n".join(lines)))

| setting | field | 302× | 142× |
|---|---|---|---|
| atom equivalents | `n_equiv` | 4337 | 9216 |
| field fit steps | `fit.steps` | 1000 | 1000 |
| polish steps | `fit.polish_steps` | 2 | 2 |
| potential fit steps | `physics.epochs` | 40 | 40 |
| Gabor splats per group | `carriers.tied_k` | 7 | 7 |
| carrier seed modes | `carriers.tail_bins` | [1, 6, 11, 16, 21, 26, 31] | [1, 6, 11, 16, 21, 26, 31] |
| group width along y (cells) | `carriers.sigma_y_cells` | 1.5 | 1.5 |
| joint fit steps | `carriers.routed_steps` | 60 | 60 |
| joint fit steps, placement-bound | `carriers.routed_steps_hard` | 120 | 120 |
| gradient-ratio threshold | `carriers.hard_ratio` | 50.0 | 50.0 |
| flux gate | `carriers.flux_gate` | 5.0 | 5.0 |
| amplitude ridge (x mean Gram diagonal) | `varpro.ridge` | 0.001 | 0.001 |
| Gauss-Newton steps | `gn.steps` | 8 | 8 |
| Gauss-Newton damping | `gn.mu` | 0.0001 | 0.0001 |
| Gauss-Newton on the Gabor amplitudes only | `gn.carriers_only` | True | True |
| refined codes: Adam steps | `baselines.objective_warmup` | 400 | 400 |
| refined codes: damping (x max diag JJ^T) | `baselines.objective_relative_damping` | 1e-08 | 1e-08 |
| flux modes | `band` | [1, 32] | [1, 32] |

## Final-fit checks (Table 22)

The stored GyroSplat fits, 39 snapshots per budget (seed 0): median and worst flux error, the count below 1e-4, net-flux sign reversals, the count with κ / κ_truth within 10%, and the lowest potential PSNR.

In [3]:
display(Markdown(X.final_fit_table(rows)))

| CR | median flux | worst flux | < 1e-4 | sign flips | κ within 10% | min PSNR_φ |
|---|---|---|---|---|---|---|
| 302× | 5.1×10⁻⁷ | 3.6×10⁻⁵ | 39/39 | 0/39 | 36/39 | 21.45 |
| 142× | 4.5×10⁻⁸ | 1.6×10⁻⁶ | 39/39 | 0/39 | 38/39 | 23.71 |

## Damping and step count of the final refinement (Table 23)

Every row starts from the same saved joint-fit states of 12 snapshots at 302× (including the hardest ones). Flux columns: median and worst after the refinement; the potential-PSNR change is against the saved state. The recipe uses damping 1e-4 and eight steps. From 1e-8 to 1e-4 the result hardly changes; from 1e-3 on, eight steps are too few. The hardest snapshot loses about 1 dB of potential PSNR whenever its flux is fitted, at any damping.

In [4]:
display(Markdown(X.damping_table(exp)))

| damping | steps | median flux | worst flux | median ΔPSNR_φ | worst ΔPSNR_φ |
|---|---|---|---|---|---|
| 1.0×10⁻⁸ | 8 | 2.0×10⁻⁸ | 5.6×10⁻⁸ | -0.005 | -1.057 |
| 1.0×10⁻⁶ | 8 | 2.6×10⁻⁸ | 5.8×10⁻⁸ | -0.005 | -1.054 |
| 1.0×10⁻⁵ | 8 | 2.8×10⁻⁸ | 5.6×10⁻⁸ | -0.005 | -1.033 |
| 1.0×10⁻⁴ | 1 | 4.3×10⁻⁴ | 0.022 | -0.004 | -0.848 |
| 1.0×10⁻⁴ | 2 | 1.6×10⁻⁴ | 0.005 | -0.005 | -0.919 |
| 1.0×10⁻⁴ | 4 | 2.7×10⁻⁵ | 7.5×10⁻⁴ | -0.005 | -0.954 |
| 1.0×10⁻⁴ | 8 | 5.7×10⁻⁷ | 3.6×10⁻⁵ | -0.005 | -0.957 |
| 1.0×10⁻⁴ | 16 | 6.0×10⁻⁸ | 2.0×10⁻⁷ | -0.005 | -0.957 |
| 0.0010 | 8 | 3.0×10⁻⁴ | 0.010 | -0.004 | -0.807 |
| 0.0100 | 8 | 0.001 | 0.069 | -0.001 | -0.394 |
| 0.1000 | 8 | 0.001 | 0.141 | -0.000 | -0.056 |

## Time and memory (Tables 24 and 25)

One job per dedicated GPU (NVIDIA B300), six CPU threads, methods taking turns, each GPU's first job and the jobs that shared their GPU with another process discarded; nine pilot snapshots and two seeds. Encoding is the complete fit; decoding renders the full 5D distribution. Median [interquartile range].

In [5]:
display(Markdown(X.timing_table(exp)))
display(Markdown(X.stage_table(exp)))

| method | CR | fits | encode (s) | decode (s) | peak GB |
|---|---|---|---|---|---|
| GyroSplat | 302× | 18 | 169 [167–172] | 0.042 [0.0419–0.042] | 19.0 |
| NF + physics | 302× | 18 | 85.1 [82.6–88.6] | 0.166 [0.155–0.201] | 55.0 |
| NF, field only | 302× | 18 | 49.4 [47.7–51.4] | 0.364 [0.322–0.513] | 1.2 |
| Splats, field only | 302× | 18 | 114 [113–115] | — | 5.1 |
| GyroSplat | 142× | 18 | 399 [399–401] | 0.0884 [0.0883–0.0884] | 39.1 |
| NF + physics | 142× | 17 | 116 [111–119] | 0.245 [0.24–0.29] | 94.2 |
| NF, field only | 142× | 17 | 63.6 [59.4–65.3] | 0.372 [0.335–0.506] | 1.2 |
| Splats, field only | 142× | 18 | 237 [236–238] | — | 10.3 |

| stage | 302× (s) | 142× (s) |
|---|---|---|
| Distribution fit | 87 | 184 |
| Potential fit | 6 | 10 |
| Gabor + joint fit | 64 | 182 |
| Final GN refinement | 11 | 24 |
| Total encoding | 169 | 399 |

In [6]:
ops = X.operator_cost(exp)
print(f"diagnostic operator (geometry and reference readouts): {ops['construct_seconds']:.2f} s, "
      f"{ops['construct_peak_gb']:.1f} GB peak")
print(f"held while encoding: reference arrays {ops['reference_bytes'] / 1e6:.0f} MB, geometry {ops['geometry_bytes'] / 1e6:.2f} MB "
      f"(shared by the snapshots of a trajectory)")
for tier in (302, 142):
    print(f"Gauss-Newton Jacobian at {tier}x: 31 modes x 2 x {ops[f'gabor_amplitudes_{tier}']:,.0f} Gabor amplitudes, "
          f"float64 = {ops[f'jacobian_bytes_{tier}'] / 1e6:.2f} MB")

diagnostic operator (geometry and reference readouts): 2.51 s, 0.9 GB peak
held while encoding: reference arrays 223 MB, geometry 3.61 MB (shared by the snapshots of a trajectory)
Gauss-Newton Jacobian at 302x: 31 modes x 2 x 3,682 Gabor amplitudes, float64 = 1.83 MB
Gauss-Newton Jacobian at 142x: 31 modes x 2 x 7,833 Gabor amplitudes, float64 = 3.89 MB


None of this is stored with the code: decoding needs only the stored parameters, and the diagnostics need the trajectory's geometry, which every method needs.